In [ ]:
import gravis as gv
import networkx as nx

from nomad_utility_workflows.utils.workflows import (
    NodeAttributes,
    NodeAttributesUniverse,
    build_nomad_workflow,
    nodes_to_graph,
)

In [ ]:
node_attributes = {
    0: NodeAttributes(
        name='input system',
        type='input',
        path_info={
            'mainfile_path': 'Emin/mdrun_Emin.log',
            'supersection_index': 0,
            'section_index': 0,
            'section_type': 'system',
        },
        out_edge_nodes=[1],
    ),
    1: NodeAttributes(
        name='Geometry Optimization',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Emin/mdrun_Emin.log'},
    ),
    2: NodeAttributes(
        name='Equilibration NPT Molecular Dynamics',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Equil_NPT/mdrun_Equil-NPT.log'},
        in_edge_nodes=[1],
    ),
    3: NodeAttributes(
        name='Production NVT Molecular Dynamics',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log'},
        in_edge_nodes=[2],
    ),
    4: NodeAttributes(
        name='output system',
        type='output',
        path_info={
            'section_type': 'system',
            'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log',
        },
        in_edge_nodes=[3],
    ),
    5: NodeAttributes(
        name='output properties',
        type='output',
        path_info={
            'section_type': 'calculation',
            'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log',
        },
        in_edge_nodes=[3],
    ),
}

node_attributes_universe = NodeAttributesUniverse(nodes=node_attributes)

In [ ]:
workflow_graph_input_minimal = nodes_to_graph(node_attributes_universe)

gv.d3(
    workflow_graph_input_minimal,
    node_label_data_source='name',
    edge_label_data_source='name',
    zoom_factor=1.5,
    node_hover_tooltip=True,
)

In [ ]:
for node_key, node_attributes in workflow_graph_input_minimal.nodes(data=True):
    if 'in_edge_nodes' in node_attributes:
        del node_attributes['in_edge_nodes']
    if 'out_edge_nodes' in node_attributes:
        del node_attributes['out_edge_nodes']

for node_key, node_attributes in list(workflow_graph_input_minimal.nodes(data=True)):
    print(node_key, node_attributes)

In [ ]:
for edge_1, edge_2, edge_attributes in workflow_graph_input_minimal.edges(data=True):
    print(edge_1, edge_2, edge_attributes)

In [ ]:
workflow_metadata = {
    'destination_filename': './workflow_minimal.archive.yaml',
    'workflow_name': 'Equilibration Procedure',
}

workflow_graph_output_minimal = build_nomad_workflow(
    workflow_metadata=workflow_metadata,
    workflow_graph=nx.DiGraph(workflow_graph_input_minimal),
    write_to_yaml=True,
)

gv.d3(
    workflow_graph_output_minimal,
    node_label_data_source='name',
    edge_label_data_source='name',
    zoom_factor=1.5,
    node_hover_tooltip=True,
)

In [ ]:
for node_key, node_attributes in list(workflow_graph_output_minimal.nodes(data=True)):
    print(node_key, node_attributes)

In [ ]:
for edge_1, edge_2, edge_attributes in workflow_graph_output_minimal.edges(data=True):
    print(edge_1, edge_2, edge_attributes)

In [ ]:
# Now let's add some additional input/outputs to the nodes

node_attributes = {
    0: NodeAttributes(
        name='input system',
        type='input',
        path_info={
            'mainfile_path': 'Emin/mdrun_Emin.log',
            'supersection_index': 0,
            'section_index': 0,
            'section_type': 'system',
        },
        out_edge_nodes=[1],
    ),
    1: NodeAttributes(
        name='Geometry Optimization',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Emin/mdrun_Emin.log'},
        outputs=[
            {
                'name': 'energies of the relaxed system',
                'path_info': {
                    'section_type': 'energy',
                    'supersection_path': 'run/0/calculation',  # this can be done,
                    # but at this point it's safer / easier to just use archive_path
                    'supersection_index': -1,
                },
            }
        ],
    ),
    2: NodeAttributes(
        name='Equilibration NPT Molecular Dynamics',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Equil_NPT/mdrun_Equil-NPT.log'},
        in_edge_nodes=[1],
        outputs=[
            {
                'name': 'MD workflow properties (structural and dynamical)',
                'path_info': {
                    'section_type': 'results',
                },
            }
        ],
    ),
    3: NodeAttributes(
        name='Production NVT Molecular Dynamics',
        type='workflow',
        entry_type='simulation',
        path_info={'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log'},
        in_edge_nodes=[2],
        outputs=[
            {
                'name': 'MD workflow properties (structural and dynamical)',
                'path_info': {
                    'section_type': 'results',
                },
            }
        ],
    ),
    4: NodeAttributes(
        name='output system',
        type='output',
        path_info={
            'section_type': 'system',
            'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log',
        },
        in_edge_nodes=[3],
    ),
    5: NodeAttributes(
        name='output properties',
        type='output',
        path_info={
            'section_type': 'calculation',
            'mainfile_path': 'Prod_NVT/mdrun_Prod-NVT.log',
        },
        in_edge_nodes=[3],
    ),
}

node_attributes_universe = NodeAttributesUniverse(nodes=node_attributes)

In [ ]:
workflow_graph_input = nodes_to_graph(node_attributes_universe)

gv.d3(
    workflow_graph_input,
    node_label_data_source='name',
    edge_label_data_source='name',
    zoom_factor=1.5,
    node_hover_tooltip=True,
)

In [ ]:
for node_key, node_attributes in workflow_graph_input.nodes(data=True):
    print(node_key, node_attributes)

In [ ]:
for edge_1, edge_2, edge_attributes in workflow_graph_input.edges(data=True):
    print(edge_1, edge_2, edge_attributes)

In [ ]:
workflow_metadata = {
    'destination_filename': './workflow.archive.yaml',
    'workflow_name': 'Equilibration Procedure',
}

workflow_graph_output = build_nomad_workflow(
    workflow_metadata=workflow_metadata,
    workflow_graph=nx.DiGraph(workflow_graph_input),
    write_to_yaml=True,
)

gv.d3(
    workflow_graph_output,
    node_label_data_source='name',
    edge_label_data_source='name',
    zoom_factor=1.5,
    node_hover_tooltip=True,
)

In [ ]:
for node_key, node_attributes in workflow_graph_output.nodes(data=True):
    print(node_key, node_attributes)

In [ ]:
for edge_1, edge_2, edge_attributes in workflow_graph_output.edges(data=True):
    print(edge_1, edge_2, edge_attributes)